In [19]:
!sudo apt-get install -y pciutils lshw zstd
!curl -fsSL https://ollama.com/install.sh | sh
! sudo systemctl restart ollama
! ollama --version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.7.0-6).
zstd is already the newest version (1.4.8+dfsg-3build1).
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down
ollama version is 0.23.2


In [20]:
import subprocess, time, os, requests

  # Kill existing ollama serve
subprocess.run(['pkill', '-f', 'ollama serve'], capture_output=True)
time.sleep(2)

  # Restart with origins open
env = os.environ.copy()
env['OLLAMA_ORIGINS'] = '*'

proc = subprocess.Popen(
      ['ollama', 'serve'],
      stdout=subprocess.DEVNULL,
      stderr=subprocess.DEVNULL,
      env=env
)
time.sleep(3)

  # Verify
r = requests.get("http://localhost:11434/api/tags")
print("✓ Ollama running:", r.status_code)

✓ Ollama running: 200


In [21]:
!ollama pull qwen2.5:7b

In [22]:
import requests, json
r = requests.post("http://localhost:11434/api/generate",
      json={"model": "qwen2.5:7b", "prompt": "hi", "stream": False})
print(r.json())

{'model': 'qwen2.5:7b', 'created_at': '2026-05-11T07:52:32.266387657Z', 'response': 'Hello! How can I assist you today? Feel free to ask me any questions or let me know if you need help with anything specific.', 'done': True, 'done_reason': 'stop', 'context': [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 6023, 151645, 198, 151644, 77091, 198, 9707, 0, 2585, 646, 358, 7789, 498, 3351, 30, 31733, 1910, 311, 2548, 752, 894, 4755, 476, 1077, 752, 1414, 421, 498, 1184, 1492, 448, 4113, 3151, 13], 'total_duration': 4220032079, 'load_duration': 3338025571, 'prompt_eval_count': 30, 'prompt_eval_duration': 86363683, 'eval_count': 29, 'eval_duration': 695463244}


In [2]:
import subprocess
subprocess.run(['fuser', '-k', '11435/tcp'], capture_output=True)


CompletedProcess(args=['fuser', '-k', '11435/tcp'], returncode=1, stdout=b'', stderr=b'')

In [3]:
import threading
from http.server import HTTPServer, BaseHTTPRequestHandler
import urllib.request

class OllamaProxy(BaseHTTPRequestHandler):
      def do_request(self):
          length = int(self.headers.get('Content-Length', 0))
          body = self.rfile.read(length) if length > 0 else None
          headers = {k: v for k, v in self.headers.items()
                     if k.lower() not in ('host', 'content-length')}
          if body:
              headers['Content-Length'] = str(len(body))
          try:
              req = urllib.request.Request(
                  f'http://localhost:11434{self.path}',
                  data=body, headers=headers, method=self.command
              )
              with urllib.request.urlopen(req, timeout=300) as resp:
                  self.send_response(resp.status)
                  for k, v in resp.headers.items():
                      if k.lower() != 'transfer-encoding':
                          self.send_header(k, v)
                  self.end_headers()
                  while chunk := resp.read(4096):
                      self.wfile.write(chunk)
          except Exception as e:
              self.send_error(502, str(e))

      do_GET = do_POST = do_PUT = do_DELETE = do_request
      def log_message(self, *_): pass

server = HTTPServer(('0.0.0.0', 11435), OllamaProxy)
threading.Thread(target=server.serve_forever, daemon=True).start()
print("✓ Proxy on :11435")

✓ Proxy on :11435


In [14]:
! curl -L https://github.com/ekzhang/bore/releases/download/v0.5.0/bore-v0.5.0-x86_64-unknown-linux-musl.tar.gz -o /tmp/bore.tar.gz && tar -xzf /tmp/bore.tar.gz -C /usr/local/bin && chmod +x /usr/local/bin/bore && bore --version

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2007k  100 2007k    0     0  3209k      0 --:--:-- --:--:-- --:--:-- 3209k
bore-cli 0.5.0


In [15]:
import subprocess, threading, time

proc = subprocess.Popen(
      ['bore', 'local', '11435', '--to', 'bore.pub'],
      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
  )

def read():
      for line in proc.stdout:
          print(line, end='')

threading.Thread(target=read, daemon=False).start()
time.sleep(10)


2026-05-11T08:03:26.224466Z  INFO bore_cli::client: connected to server remote_port=27425
2026-05-11T08:03:26.224530Z  INFO bore_cli::client: listening at bore.pub:27425


In [17]:
import subprocess, time, os, requests

subprocess.run(['pkill', '-f', 'ollama serve'], capture_output=True)
time.sleep(2)

env = os.environ.copy()
env['OLLAMA_ORIGINS'] = '*'

proc = subprocess.Popen(
      ['ollama', 'serve'],
      stdout=subprocess.DEVNULL,
      stderr=subprocess.DEVNULL,
      env=env
)
time.sleep(3)

r = requests.get("http://localhost:11434/api/tags")
print("✓ Ollama running:", r.status_code)


✓ Ollama running: 200


In [18]:
import requests, subprocess, re

  # Check Ollama
try:
      r = requests.get("http://localhost:11434/api/tags", timeout=5)
      print("✓ Ollama alive:", r.status_code)
except Exception as e:
      print("✗ Ollama down:", e)

  # Check proxy on :11435
try:
      r = requests.get("http://localhost:11435/api/tags", timeout=5)
      print("✓ Proxy on :11435 alive:", r.status_code)
except Exception as e:
      print("✗ Proxy on :11435 down:", e)

  # Check bore process + extract port
result = subprocess.run(['pgrep', '-a', '-f', 'bore local'], capture_output=True, text=True)
if result.stdout.strip():
      print("✓ bore running:", result.stdout.strip())
      m = re.search(r'bore local (\d+)', result.stdout)
      port = m.group(1) if m else "unknown"
      print(f"  Listening on: bore.pub (local port {port})")
else:
      print("✗ bore NOT running")

  # Verify bore tunnel reachable — update port below
BORE_URL = "http://bore.pub:27425"  # ← update this each session
try:
      r2 = requests.get(f"{BORE_URL}/api/tags", timeout=10)
      print("✓ Bore tunnel reachable:", r2.status_code, BORE_URL)
except Exception as e:
      print("✗ Bore tunnel unreachable:", e)

  # Check OLLAMA_ORIGINS
result = subprocess.run(['pgrep', '-f', 'ollama serve'], capture_output=True, text=True)
pid = result.stdout.strip().split('\n')[0]
if pid:
      env_bytes = open(f'/proc/{pid}/environ', 'rb').read()
      env_str = env_bytes.replace(b'\x00', b'\n').decode(errors='ignore')
      print("OLLAMA_ORIGINS set:", 'OLLAMA_ORIGINS' in env_str)
      print("Value:", [l for l in env_str.split('\n') if 'OLLAMA_ORIGINS' in l])

✓ Ollama alive: 200
✓ Proxy on :11435 alive: 200
✓ bore running: 21399 bore local 11435 --to bore.pub
  Listening on: bore.pub (local port 11435)
2026-05-11T08:09:49.607621Z  INFO proxy{id=efbb0478-13ca-46a5-a8c6-bb345906f9e7}: bore_cli::client: new connection
2026-05-11T08:09:50.069881Z  INFO proxy{id=efbb0478-13ca-46a5-a8c6-bb345906f9e7}: bore_cli::client: connection exited
✓ Bore tunnel reachable: 200 http://bore.pub:27425
OLLAMA_ORIGINS set: True
Value: ['OLLAMA_ORIGINS=*']


In [19]:
import time, requests
print("Keep-alive running — leave this cell running")
while True:
      try: requests.get("http://localhost:11434/api/tags")
      except: pass
      time.sleep(30)

Keep-alive running — leave this cell running
2026-05-11T08:12:25.735461Z  INFO proxy{id=c84736ba-0bbc-417f-89d0-10de78fbad45}: bore_cli::client: new connection
2026-05-11T08:12:40.331532Z  INFO proxy{id=c84736ba-0bbc-417f-89d0-10de78fbad45}: bore_cli::client: connection exited
2026-05-11T08:12:43.504719Z  INFO proxy{id=3c40677f-83e0-4569-b8aa-b1f1c1a4dd56}: bore_cli::client: new connection
2026-05-11T08:12:45.536313Z  INFO proxy{id=3c40677f-83e0-4569-b8aa-b1f1c1a4dd56}: bore_cli::client: connection exited
2026-05-11T08:18:31.839001Z  INFO proxy{id=53d7d5a7-f268-489a-a01d-a50c927eb19d}: bore_cli::client: new connection
2026-05-11T08:18:34.603210Z  INFO proxy{id=53d7d5a7-f268-489a-a01d-a50c927eb19d}: bore_cli::client: connection exited
2026-05-11T08:18:35.569226Z  INFO proxy{id=4961ef17-c590-45ef-af51-ec7e9df456ab}: bore_cli::client: new connection
2026-05-11T08:18:37.033657Z  INFO proxy{id=4961ef17-c590-45ef-af51-ec7e9df456ab}: bore_cli::client: connection exited
2026-05-11T08:18:37.823

KeyboardInterrupt: 